In [ ]:
import re
import pandas as pd

# 1. Leer el archivo original de 2023-2022 como texto plano
with open('llegadas_2023_2022_clean.csv', 'r', encoding='utf-8-sig', errors='ignore') as f:
    lineas = f.readlines()

lineas_datos = [l.strip() for l in lineas if l.strip() != ""]
if "RO" in lineas_datos[0]:
    lineas_datos = lineas_datos[1:]

# 2. ORDEN INVERTIDO DE LAS COLUMNAS (De más nuevo a más viejo: Dic 2023 -> Ene 2022)
meses_columnas = [
    "2023-12", "2023-11", "2023-10", "2023-09", "2023-08", "2023-07",
    "2023-06", "2023-05", "2023-04", "2023-03", "2023-02", "2023-01",
    "2022-12", "2022-11", "2022-10", "2022-09", "2022-08", "2022-07",
    "2022-06", "2022-05", "2022-04", "2022-03", "2022-02", "2022-01"
]

matriz_paises = {}

# 3. Extraer la serie temporal de cada país
for linea in lineas_datos[1:]:
    if "Pasajeros Totales" in linea:
        continue

    paises_encontrados = re.findall(r'[A-ZÁÉÍÓÚÑüÜ]{3,}(?:\s+[A-ZÁÉÍÓÚÑüÜ]+)*', linea)
    if not paises_encontrados:
        continue
    pais = paises_encontrados[0].strip()

    # Capturar bloques numéricos
    numeros = re.findall(r'"([\d\.]+)"', linea)
    if not numeros:
        numeros = re.findall(r'\b\d+(?:\.\d+)*\b', linea)

    valores_meses = []
    for num in numeros:
        # Si el número contiene un punto y termina con exactamente dos dígitos.
        # significa que se comieron el 0 final al escribirlo. Se lo añadimos.
        if '.' in num and len(num.split('.')[-1]) == 2:
            num = num + "0"
        # ------------------------------------------------

        num_limpio = num.replace('.', '')
        if num_limpio.isdigit():
            valores_meses.append(int(num_limpio))

    matriz_paises[pais] = valores_meses

# 4. Construir la estructura temporal mes a mes
paises_top = {'ESPAÑA', 'SPAIN', 'FRANCIA', 'FRANCE', 'ITALIA', 'ITALY',
              'REINO UNIDO', 'UNITED KINGDOM', 'UK', 'PAISES BAJOS', 'HOLANDA',
              'NETHERLANDS', 'ALEMANIA', 'GERMANY'}

datos_historicos_finales = []

for idx_mes, mes_nombre in enumerate(meses_columnas):

    def extraer_valor_mes(nombre_pais):
        for k, lista_valores in matriz_paises.items():
            if nombre_pais in k and idx_mes < len(lista_valores):
                return lista_valores[idx_mes]
        return 0

    nac = extraer_valor_mes('ESPAÑA') or extraer_valor_mes('SPAIN')
    fr = extraer_valor_mes('FRANCIA') or extraer_valor_mes('FRANCE')
    it = extraer_valor_mes('ITALIA') or extraer_valor_mes('ITALY')
    uk = extraer_valor_mes('REINO UNIDO') or extraer_valor_mes('UNITED KINGDOM') or extraer_valor_mes('UK')
    nl = extraer_valor_mes('PAISES BAJOS') or extraer_valor_mes('HOLANDA') or extraer_valor_mes('NETHERLANDS')
    de = extraer_valor_mes('ALEMANIA') or extraer_valor_mes('GERMANY')

    resto_del_mundo = 0
    for k, lista_valores in matriz_paises.items():
        if not any(p in k for p in paises_top) and idx_mes < len(lista_valores):
            resto_del_mundo += lista_valores[idx_mes]

    internacionales = 0
    for k, lista_valores in matriz_paises.items():
        if 'ESPAÑA' not in k and 'SPAIN' not in k and idx_mes < len(lista_valores):
            internacionales += lista_valores[idx_mes]

    total_aeropuerto = nac + internacionales

    datos_historicos_finales.append({
        'Mes': mes_nombre,
        'Volumen total llegadas aeropuerto': total_aeropuerto,
        'Volumen llegadas Nacionales': nac,
        'Volumen llegadas Internacionales': internacionales,
        'Volumen llegadas Francia': fr,
        'Volumen llegadas Italia': it,
        'Volumen llegadas Reino Unido': uk,
        'Volumen llegadas Países Bajos': nl,
        'Volumen llegadas Alemania': de,
        'Volumen llegadas Resto del Mundo': resto_del_mundo
    })

# 5. Convertir a DataFrame, ordenar cronológicamente (Ene 2022 -> Dic 2023) y guardar
df_temporal = pd.DataFrame(datos_historicos_finales)
df_temporal = df_temporal.sort_values(by='Mes').reset_index(drop=True)

df_temporal.to_csv('2023-2022.csv', index=False, encoding='utf-8-sig')


¡Arreglado! Salvado como '2023-2022.csv'. Ahora Enero de 2022 para Alemania marcará exactamente 383160.


,Mes,Volumen total llegadas aeropuerto,Volumen llegadas Nacionales,Volumen llegadas Internacionales,Volumen llegadas Francia,Volumen llegadas Italia,Volumen llegadas Reino Unido,Volumen llegadas Países Bajos,Volumen llegadas Alemania,Volumen llegadas Resto del Mundo
0,2022-01,5049920,2071925,2977995,256898,253528,426544,202674,383160,1455191
1,2022-02,6112444,2281382,3831062,342496,338632,771780,236290,513491,1628373
2,2022-03,7893086,2854079,5039007,403646,451448,1070650,299066,741636,2072561
3,2022-04,10354994,3368893,6986101,598636,597323,1601828,391295,1123338,2673681
4,2022-05,11327019,3569286,7757733,638886,646297,1980236,432738,1217537,2842039
5,2022-06,12177179,3894518,8282661,648866,717195,2080070,426534,1323147,3086849
6,2022-07,13671949,4389171,9282778,763450,812543,2300487,427839,1374565,3603894
7,2022-08,13432976,4416528,9016448,712275,834716,2240484,409589,1340428,3478956
8,2022-09,12491646,3997026,8494620,638293,740338,2048072,417454,1309077,3341386
9,2022-10,11818547,3855030,7963517,678933,689233,1824985,396575,1214660,3159131
